## 파이토치로 다층 퍼셉트론 구현하기

In [ ]:
from IPython.display import Image
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/9-2.DNN.png', width=500)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
torch.cuda.is_available()

In [ ]:
# GPU 연산이 가능하다면 GPU를 사용하고, 불가능하다면 CPU를 사용
# GPU와 CPU 차이: https://youtu.be/1BAZf3PsjWA?si=YSXlHCODtRKguvto

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

### 1) 다층 퍼셉트론을 이용한 XOR 문제 풀기

**`.to(device)`**: 텐서나 모델을 지정한 장치(GPU 또는 CPU)로 보냄

* 위에서 `device`를 GPU를 쓸 수 있으면 `'cuda'`, 아니면 `'cpu'`로 정해 두었다.
* 계산에 함께 쓰이는 **데이터와 모델은 같은 장치에 있어야** 한다. 서로 다른 장치에 있으면 에러가 발생한다.
* 그래서 아래에서 데이터(`X`, `Y`)와 모델을 만들 때 모두 `.to(device)`를 붙인다.

In [ ]:
X = torch.FloatTensor([[0,0],[0,1],[1,0],[1,1]]).to(device)
Y = torch.FloatTensor([[0],[1],[1],[0]]).to(device)

In [ ]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/11-1.DNN.png', width=500)

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), # input_layer = 2, hidden_layer1 = 10
    nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), # hidden_layer1 = 10, hidden_layer2 = 10
    nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), # hidden_layer2 = 10, hidden_layer3 = 10
    nn.Sigmoid(),
    nn.Linear(10, 1, bias=True), # hidden_layer3 = 10, output_layer = 1
    nn.Sigmoid()
).to(device)

In [ ]:
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

**`F.binary_cross_entropy` vs `nn.BCELoss`**

6-2에서는 `F.binary_cross_entropy(hypothesis, y)`를 썼는데, 여기서는 `nn.BCELoss()`를 쓴다. **계산 결과는 완전히 같다.** 차이는 사용하는 방식뿐이다.

| | `torch.nn.functional` (F) | `torch.nn` |
|---|---|---|
| 형태 | 함수 | 클래스 (객체를 먼저 만든 뒤 호출) |
| 사용 | `loss = F.binary_cross_entropy(h, y)` | `criterion = nn.BCELoss()` → `loss = criterion(h, y)` |
| 비슷한 예 | `F.mse_loss`, `F.cross_entropy` | `nn.MSELoss`, `nn.CrossEntropyLoss` |

* `nn.BCELoss()`는 내부에서 `F.binary_cross_entropy`를 호출한다.
* 일반적으로는 클래스 형태(`nn.BCELoss`, `nn.CrossEntropyLoss` 등)를 많이 쓴다.

In [ ]:
import time

start = time.time()
for epoch in range(10001):
    optimizer.zero_grad()
    hypothesis = model(X)

    # 비용 함수 계산 및 역전파
    cost = criterion(hypothesis, Y)
    cost.backward()
    optimizer.step()

    # 100 epoch마다 loss 출력
    if epoch % 100 == 0:
        print(f'Epoch {epoch}/10000 Cost: {cost.item():.6f}')
print(f"{device} time: {time.time() - start:.3f} sec")

In [ ]:
X

In [ ]:
with torch.no_grad():
    hypothesis = model(X)
    predicted = (hypothesis > 0.5).float()
    accuracy = (predicted == Y).float().mean()
    print('모델의 출력값(Hypothesis): \n', hypothesis)
    print('모델의 예측값(Predicted): \n', predicted)
    print('실제값(Y): \n', Y)
    print('정확도(Accuracy): \n', accuracy.item())


* ```with torch.no_grad()```: 기울기 계산을 비활성화 하여 연산 속도 증가 및 메모리 사용량 감소

### 2) 다층 퍼셉트론으로 MNIST 분류하기

#### MNIST 데이터

* MNIST: 숫자 0부터 9까지의 이미지로 구성된 손글씨 데이터셋
    - 6만개의 훈련 데이터와 레이블, 1만개의 테스트 데이터와 레이블로 구성
    - 레이블: 0 부터 9 (10개)

In [ ]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/11-1.mnist.png', width=500)

In [ ]:
# 각각의 이미지는 28x28 크기의 픽셀
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/11-1.mnist_input.png', width=500)

In [ ]:
# 다층 퍼셉트론의 input으로 사용하기 위해 28x28 크기의 2차원 배열을 784 크기의 1차원 배열로 변환
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/11-1.mnist_mat_to_vec.png', width=500)

## 코드 참고
# for X, Y in data_loader:
#     X = X.view(-1, 28*28)

#### MNIST 분류기 구현

##### i) 사전 설정 

In [ ]:
import torch

In [ ]:
# GPU 연산이 가능하다면 GPU를 사용하고, 불가능하다면 CPU를 사용
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("다음 기기로 학습합니다:", device)


In [ ]:
# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

##### ii) MNIST 데이터 로드

--------------------
**DataLoader**

In [ ]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

TensorDataset은 기본적으로 텐서를 입력으로 받음. 텐서 형태로 데이터 정의.

In [ ]:
x_train  =  torch.FloatTensor([[73,  80,  75], 
                               [93,  88,  93], 
                               [89,  91,  90], 
                               [96,  98,  100],   
                               [73,  66,  70]])  
y_train  =  torch.FloatTensor([[152],  [185],  [180],  [196],  [142]])

dataset = TensorDataset(x_train, y_train)

In [ ]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

데이터로더는 `for` 문으로 미니 배치를 하나씩 차례대로 꺼낼 수 있음. 데이터가 5개이고 `batch_size=2`이므로 2개, 2개, 1개씩 나옴. (셀을 다시 실행하면 `shuffle=True` 때문에 순서가 바뀜)

In [ ]:
for x, y in dataloader:
    print('x:', x)
    print('y:', y)
    print('----------')

* ```batch_size```: 미니 배치의 크기
* ```shuffle```: Epoch마다 데이터셋을 섞어 데이터가 학습되는 순서를 바꿈. 순서가 학습에 영향을 주는 것을 방지
* ```drop_last```: 마지막 배치는 다른 미니 배치보다 개수가 적을 수 있음. 이 미니 배치를 버릴지 결정

In [ ]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, drop_last=True)

In [ ]:
for x, y in dataloader:
    print('x:', x)
    print('y:', y)
    print('----------')

--------------------------------

In [ ]:
## Import torchvision
import torchvision.datasets as datasets

* ```torchvision```: 유명한 데이터셋들, 이미 구현되어져 있는 유명한 모델들, 일반적인 이미지 전처리 도구들을 포함하고 있는 패키지

In [ ]:

# MNIST dataset
mnist_train = datasets.MNIST(root="Data",              # 데이터가 저장될 경로
                            train=True,                         # True면 훈련 데이터, False면 테스트 데이터
                            download=True)                      # 데이터가 없다면 다운로드 (데이터가 있는 경우 설정 필요 X)

In [ ]:
mnist_train.data[0]

In [ ]:
mnist_train.data[0].shape

In [ ]:
mnist_train.targets

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
img = mnist_train.data[0]
plt.imshow(img, cmap='gray')

In [ ]:
X = mnist_train.data.view(-1, 28*28) / 255.0 # 정규화, matrix to vector
y = mnist_train.targets

In [ ]:
train_dataset = TensorDataset(X, y)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

##### iii) 다층 퍼셉트론 설계 및 학습

784(28x28)개의 입력 뉴런, 10개의 출력 뉴런(0~9)을 가지는 다층 퍼셉트론을 구성

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
model = nn.Sequential(
    nn.Linear(28**2, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

print(model)

In [ ]:
# 오차함수 선택
loss_fn = nn.CrossEntropyLoss().to(device) # 내부적으로 softmax 포함

# 최적화 기법 선택
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# hyperparameters
num_epochs = 3

for epoch in range(num_epochs):
    for data, targets in train_dataloader:
        optimizer.zero_grad()
        y_pred = model(data.to(device))
        loss = loss_fn(y_pred, targets.to(device))
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

##### iv) 테스트 데이터에 대해 평가

In [ ]:
mnist_test = datasets.MNIST(root="Data",
                           train=False,
                           download=True)

X_test = mnist_test.data.view(-1, 28*28) / 255.0
y_test = mnist_test.targets

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, drop_last=False)

In [ ]:
model.eval()    # 신경망을 추론(학습 없이 예측만 하는 것) 모드로 전환

correct = 0

# 데이터로더에서 미니배치를 하나씩 꺼내 추론을 수행
with torch.no_grad():  # 추론 과정에는 미분이 필요없음
    for data, targets in test_dataloader:
        outputs = model(data.to(device))  # 데이터를 입력하고 출력을 계산 (128x10)

        # 추론 계산
        _, predicted = torch.max(outputs, dim=1)  # 확률이 가장 높은 레이블이 무엇인지 계산 (argmax)
        correct += (predicted == targets.to(device)).sum().item()  # 정답과 일치한 경우 정답 카운트를 증가

# 정확도 출력
data_num = len(test_dataset)  # 데이터 총 건수
print(f'테스트 데이터에서 예측 정확도: {correct}/{data_num} ({100 * correct / data_num:.0f}%)')